# Testing class implementation

`15/09/2026`

Classes implemented in their respective modules. This notebook aims to hone
in on the API and check numerics of the calculations performed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


## 1. The potential:

In [ ]:
from emerald.potentials import MorseSoftCoulomb

msc = MorseSoftCoulomb(alpha=0.5)

In [ ]:
print(f"Well depth: {msc.well_depth}")
E = -0.5

rm, rM = msc.turning_points(E)

print(f"return points for E = {E}, r_m = {rm:.2f}, r_M = {rM:.2f}")

r_grid = np.linspace(-2, 10, 1000)

plt.plot(r_grid, msc(r_grid), label=f"{msc.name} Potential (alpha={msc.alpha})")
plt.xlabel("r")
plt.ylabel("V(r)")
plt.axhline(E, color='r', linestyle='--', label=f"E = {E}")
plt.scatter(rm, msc(rm), color='black', label=f"$r_m = {rm:.2f}$")
plt.scatter(rM, msc(rM), color='black', label=f"$r_M = {rM:.2f}$")
plt.legend()
plt.title(f"{msc.name} Potential")
plt.xlim(-2, 10)
plt.ylim(-3, 1)
plt.grid()

In [ ]:
# What other methods does Potential have?

for _E in [-1, -0.5, -0.1, 0, 0.1]:
    ps = msc.phase_space(_E, N=1000, r1=-1, r2 = 10)

    plt.plot(ps[0], ps[1], label=f"Phase Space for E = {_E}")
plt.xlabel("$r$")
plt.ylabel("$p$")
plt.legend()
plt.title(f"{msc.name} Phase Space")
plt.grid()

### 1.1 Action calculation

In [ ]:
# Action and angle calculation, let's check the convergence of the numerics:

from emerald.classical.msc_unperturbed import MsC_action

act_class = msc.action(E, method="simpson")
act_num = MsC_action(msc.alpha, E)

print(f"Action for E = {E}: class = {act_class:.10f}, script = {act_num:.10f}")

# Improve parameters to find truth value

act_class = msc.action(E, dr=1.e-7, method="simpson")
act_num = MsC_action(msc.alpha, E, dr=1.e-7)

print(f"Action for E = {E}: class = {act_class:.10f}, script = {act_num:.10f}")
#script change in the tenth decimal place. Higher class result change


In [ ]:
for dx in [1.e-5, 1.e-6, 1.e-7, 1.e-8]:
    act_num = MsC_action(msc.alpha, E, dr=dx)
    print(f"Number of points: N = {int((msc.turning_points(E)[1] - msc.turning_points(E)[0]) / dx)}")
    print(f"Action for E = {E}, dr={dx}: script = {act_num:.10f}")

In [ ]:
for N in [1e3, 1e4, 1e5, 1e6]:
    act_class = msc.action(E, N=int(N), method="simpson")
    print(f"Action for E = {E}, N={int(N)}: script = {act_class:.10f}")

In [ ]:
# timeit on class implementation



%timeit act_class = msc.action(E, dr=1.e-6, method="simpson")
%timeit act_num = MsC_action(msc.alpha, E, dr=1.e-6)


In [ ]:
%timeit act_class = msc.action(E, N=int(1e6), method="simpson")


In [ ]:
# It looks like at 1e6 points the class implementation converges and isn't as slow

Es = np.linspace(-1, -0.01, 20)

for _E in Es:
    act_class = msc.action(_E, N=int(1e6), method="simpson")
    act_num = MsC_action(msc.alpha, _E, dr=1.e-8)
    print(f"Action for E = {_E:.2f}: class = {act_class:.10f}, script = {act_num:.10f}")
    delt = abs(act_class - act_num)
    print(f"Difference: {delt:.5e}")

Okay, I'm ready to swear by the class implementation. It is more memory intensive if we choose
a tiny `dr`, but by using the number of intervals as the parameter we don't bloat the number
of evaluations as much. `N=1e6` is pretty good, reaching a maximum error of about `1.e-7`. 
Counter intuitively, "harder problems" (smaller $\alpha$) actually shrink the interval of 
integration and thus require less points (although momentum varies more abruptly there, so
I won't swear by this statement).

As a last check, let's compute action using the Gauss-Legendre quadrature:

In [ ]:
from emerald.numerics.quadrature import gauss_legendre_quadrature


def makeshift_action(E, N):
    rm, rM = msc.turning_points(E)
    action = gauss_legendre_quadrature(lambda r: msc.momentum(E, r), rm, rM, N) / np.pi
    return action

act_num = MsC_action(msc.alpha, E, dr=1.e-8)
act_class = msc.action(E, N=int(1e6), method="simpson")

act_gauss = makeshift_action(E, N=1000)
print(f"Action for E = {E}: class = {act_class:.10f}, script = {act_num:.10f}, gauss = {act_gauss:.10f}")

# Oh my, we have a contender!

In [ ]:
%timeit act_gauss = makeshift_action(E, N=1500)


In [ ]:
for _E in Es:
    act_gauss = makeshift_action(_E, N=1500)
    act_num = MsC_action(msc.alpha, _E, dr=1.e-8)
    print(f"Action for E = {_E:.2f}: gauss = {act_gauss:.10f}, script = {act_num:.10f}")
    delt = abs(act_gauss - act_num)
    print(f"Difference: {delt:.5e}")

We can safely modify the implementation to use the Gauss-Legendre quadrature method with 1500 points.
This is equivalent to approximating the integrand by a 2999 degree polynomial!


In [ ]:
msc.action(E)
# Hurray!

### 1.2 Angular frequency

Next we'll check numerical stability of angular frequency calculation, computed as
$\omega = \partial H / \partial J$: 

In [ ]:
from emerald.classical.msc_unperturbed import MsC_angular_frequency

omg_class = msc.angular_frequency(E)
omg_script = MsC_angular_frequency(msc.alpha, E)

print(f"Angular frequency for E = {E}: class = {omg_class:.10f}, script = {omg_script:.10f}")

In [ ]:
def script_omega(alpha, E, dE):

    return ( (1/(12*dE))*( MsC_action(alpha, E-2*dE,) 
                          - 8*MsC_action(alpha, E-dE,) 
                          + 8*MsC_action(alpha, E+dE,) 
                          - MsC_action(alpha, E+2*dE,) 
                          ) 
            )**(-1)

In [ ]:
for dE in [1.e-4, 1.e-5, 1.e-6, 1.e-7, 1.e-8, 1.e-9, 1.e-10, 1.e-11, 1.e-12, 1.e-13, 1.e-14, 1.e-15]:
    omg_script = script_omega(msc.alpha, E, dE)
    print(f"Frequency for E = {E}, dE={dE:.1e}: script = {omg_script:.12f}")

In [ ]:
for dE in [1.e-4, 1.e-5, 1.e-6, 1.e-7, 1.e-8, 1.e-9, 1.e-10, 1.e-11, 1.e-12, 1.e-13, 1.e-14, 1.e-15]:

    omg_class = msc.angular_frequency(E, dE)
    print(f"Frequency for E = {E}, dE={dE:.1e}: class = {omg_class:.10f}")

In [ ]:
for _E in np.linspace(-1.99, -0.01, 20):
    omg_class = msc.angular_frequency(_E, dE=1.e-6)
    omg_script = script_omega(msc.alpha, _E, dE=1.e-4)
    print(f"Frequency for E = {_E:.2f}: class = {omg_class:.10f}, script = {omg_script:.12f}")

In [ ]:
for _E in np.linspace(-1.99, -0.01, 20):
    print(f"E = {_E:.2f}, ω for dE = 1.e-5: {msc.angular_frequency(_E, dE=1.e-5):.10f}")
    print(f"E = {_E:.2f}, ω for dE = 1.e-6: {msc.angular_frequency(_E, dE=1.e-6):.10f}")
    print(f"E = {_E:.2f}, ω for dE = 1.e-7: {msc.angular_frequency(_E, dE=1.e-7):.10f} \n")

It seems that the class implementation with `dE = 1.e-6` achieves convergence within the trusty range.
Old scripted implementation is unstable because it amplifies numerical integration errors by a factor 
of $1/dE$. Finally, another way of computing $\omega(E)$ is:

$$

\omega(E) = \frac{\partial E}{\partial J} = \left ( \frac{\partial J}{\partial E} \right )^{-1} \\[10pt]
\frac{\partial J}{\partial E} = \frac{1}{\pi} \frac{\partial}{\partial E} \int_{r_m}^{r_M} p(r, E) dr
= \frac{1}{\pi} \int_{r_m}^{r_M} \frac{\partial}{\partial E} p(r, E) dr
= \frac{1}{\pi} \int_{r_m}^{r_M} \frac{1}{p(r, E)} dr
$$
so, $\omega$ will be:

$$
\omega(E) = \frac{\pi}{\int dr/p(E)}
$$

But the integrand blows up at the turning points, so we should make a change of variables:

$$
L = r_M - r_m \qquad r(\phi) = r_m + L \sin^2(\phi) \qquad dr = 2L\sin\phi \cos\phi d\phi
$$

The integral becomes:

$$
I = \int_0^{\pi/2} \frac{2L\sin\phi \cos\phi}{p(r(\phi), E)} d\phi
$$

which has a well behaved integrand at the endpoints

In [ ]:
def inv_p_transformed(phi, E):
    rm, rM = msc.turning_points(E)

    L = rM - rm
    r = rm + L * np.sin(phi)**2
    dr_dphi = 2 * L * np.sin(phi) * np.cos(phi)

    return dr_dphi / msc.momentum(E, r)

def makeshift_omega(E, N):
    I = gauss_legendre_quadrature(lambda phi: inv_p_transformed(phi, E), 0, np.pi/2, N)
    return np.pi/I

In [ ]:
for _N in [100, 500, 1000, 1500, 5000, 10000]:
    omg_makeshift = makeshift_omega(E, N=_N)
    print(f"Frequency for E = {E}, N={_N}: makeshift = {omg_makeshift:.10f}")

for _E in np.linspace(-1.99, -0.01, 20):
    omg_class = msc.angular_frequency(_E, dE=1.e-6)
    omg_makeshift = makeshift_omega(_E, N=1000)
    print(f"Frequency for E = {_E:.2f}: class = {omg_class:.10f}, makeshift = {omg_makeshift:.10f}")

In [ ]:
%timeit omg_makeshift = makeshift_omega(E, N=1000)
%timeit omg_makeshift = makeshift_omega(E, N=1500)

This is definetly the way to go and will be the standard implementation. It will also serve for angle
calculations.

### 1.3 Angle

Next we'll check numerical stability of angle calculation, computed as
the integral of $1/p$: 

In [ ]:
from emerald.classical.msc_unperturbed import MsC_angle

# MsC_angle(msc.alpha, E, rM, omg_n=msc.angular_frequency(E, dE=1.e-6))

print(f"N={1000}: \t θ(rM) = {msc.angle(E, rM, N=1000)}")
print(f"π, for reference: \t {np.pi}")

In [ ]:
msc.alpha

In [ ]:
msc = MorseSoftCoulomb(alpha=1.0)

_E = -0.5
rm, rM = msc.turning_points(_E)
r_s = np.linspace(rm, rM, 200)

for _N in [1, 2, 5, 10, 100]:
    angels = msc.angle(_E, r_s, N=_N)
    plt.plot(r_s, angels, label=f"Angle for E = {E}, N={_N}")
plt.legend()
# plt.xlim(1.7, 2)

In [ ]:
msc.turning_points(_E)

I am amazed by how eficcient the integration is. So I want to examine the integrand:

In [ ]:
for _E in np.linspace(-0.9, -0.1, 10):
    rm, rM = msc.turning_points(_E)

    phis = np.linspace(0, np.pi/2, 100)
    plt.plot(phis, msc._inv_momentum_integrand(phis, _E, rm, rM), label=f"Inv. p transformed for E = {_E:.2f}")
plt.legend()

Angle is done too. That is all we've implemented for the potential

## 2. A particle/trajectory